In [1]:
!python.exe -m pip install --upgrade pip
!pip install -q google-api-python-client pandas pymorphy2 scikit-learn;
!pip install -q pyarrow

In [2]:
import warnings
warnings.filterwarnings("ignore")
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import time
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import calendar
from datetime import date, datetime, timezone, timedelta
import os
from pathlib import Path
from dotenv import load_dotenv

In [3]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"

for p in (DATA_DIR, REPORTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

# --- БЕЗОПАСНЫЙ API КЛЮЧ ---
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("❌ Ошибка: Ключ YouTube API не найден!\n"
                     "Пожалуйста, создайте файл `.env` в корне проекта и добавьте в него строку:\n"
                     "YOUTUBE_API_KEY=ваш_ключ_из_google_cloud_console")

# Период сбора видео (можно легко поменять)
END_DATE = datetime.now(timezone.utc) - timedelta(days=2)  # запас 2 дня, чтобы избежать нулевого возраста
START_DATE = END_DATE - timedelta(days=365)

print(f"\nИнформация собрана за период c {START_DATE.strftime('%d-%m-%Y')} по {END_DATE.strftime('%d-%m-%Y')}.")

# Список из 40 каналов
channels_input = [
    '@toplesofficial', '@RedRoomLIMB', '@doctorashow', '@egor_shows', 
    '@posle_zavtra', '@newletter_unisender', '@medic_smith', '@poetische', 
    '@Alex_Tsarev', '@ganzhagleb', '@vselennayaplus', '@Scinquisitor', '@nowchpok', '@artforintrovert', 
    '@kinopoisk', '@MyGap', '@Dameoz', '@faib', '@PhysFromPobed', 
    '@ChemistryEasy', '@Thoisoi', '@cosmosprosto', '@AntropogenezRu', 
    '@postnauka', '@uch_pm', '@surdinpodcast', '@glebsolomin', 
    '@AlexanderSokolovskiy', '@VertDiderScience', '@GalileoRU', '@Igor_beletskiy', 
    '@LyutayaFizika', '@UtopiaShow', '@energolikbez', '@DoctorUtin', 
    '@dr_sychev', '@dostoverno_ru', '@irina_yakutenko', '@dysphorra', '@lankov_official'
]


Информация собрана за период c 25-10-2025 по 25-10-2026.


In [4]:
import httplib2
from googleapiclient.discovery import build

print("Создается объект youtube...")
http = httplib2.Http(disable_ssl_certificate_validation=True)
youtube = build('youtube', 'v3', developerKey=API_KEY, http=http)
print("Объект создан! Пробуем запросить каналы...")

try:
    test_req = youtube.channels().list(part='id', forHandle='@toplesofficial')
    test_res = test_req.execute()
    print("✅ УСПЕХ! API ответил. Вот данные:", test_res)
except Exception as e:
    print("❌ ОШИБКА ПРИ ЗАПРОСЕ:")
    print(e)

Создается объект youtube...
Объект создан! Пробуем запросить каналы...
✅ УСПЕХ! API ответил. Вот данные: {'kind': 'youtube#channelListResponse', 'etag': '4liXM1wU2REVCt2aWuCcabhUQLg', 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5}, 'items': [{'kind': 'youtube#channel', 'etag': 'EVv5YqS1__Rp_eUSKka9P4OYlgc', 'id': 'UC2Ru64PHqW4FxoP0xhQRvJg'}]}


In [5]:
#================================================ЭТАП 1: Получаем ID каналов==================================================
channel_ids = []

for item in channels_input:
    if item.startswith('@'):
        try:
            req = youtube.channels().list(part='id', forHandle=item)
            res = req.execute()
            if res.get('items'):
                channel_ids.append(res['items'][0]['id'])
            else:
                print(f"❌ НЕ НАЙДЕН: {item}")
        except HttpError as e:
            print(f"❌ Ошибка для {item}: {e}")

print(f"\nНайдено {len(channel_ids)} ID каналов из {len(channels_input)}.")


Найдено 40 ID каналов из 40.


In [6]:
#============================================ЭТАП 2: Сбор статистики по каналам================================================

channels_data = []

for i in range(0, len(channel_ids), 50):
    chunk = channel_ids[i:i+50]
    try:
        req = youtube.channels().list(part='snippet,statistics,contentDetails', id=','.join(chunk))
        response = req.execute()
        for item in response.get('items', []):
            stats = item['statistics']
            channels_data.append({
                'channel_id': item['id'],
                'channel_title': item['snippet']['title'],
                'subscribers': int(stats.get('subscriberCount', 0)),
                'total_views': int(stats.get('viewCount', 0)),
                'video_count': int(stats.get('videoCount', 0)),
                'uploads_playlist_id': item['contentDetails']['relatedPlaylists']['uploads']
            })
    except HttpError as e:
        print(f"❌ Ошибка при запросе каналов: {e}")

df_channels = pd.DataFrame(channels_data)

print(f'\nПо каждому каналу получена следующая информация:\nchannel_title, subscribers, total_views, video_count, uploads_playlist_id.')


По каждому каналу получена следующая информация:
channel_title, subscribers, total_views, video_count, uploads_playlist_id.


In [7]:
#================================================ЭТАП 3: Сбор статистики по видео==================================================

videos_data = []

print(f"\nСбор видео за период c {START_DATE.strftime('%d-%m-%Y')} по {END_DATE.strftime('%d-%m-%Y')}", end='\n\n')

for ch in channels_data:
    print(f"Канал {ch['channel_title']}.", end=" ")
    
    next_page_token = None
    videos_count = 0
    stop_collecting = False
    
    while not stop_collecting:
        try:
            req_items = youtube.playlistItems().list(
                part='snippet', playlistId=ch['uploads_playlist_id'], maxResults=50, pageToken=next_page_token
            )
            res_items = req_items.execute()
            video_ids = [item['snippet']['resourceId']['videoId'] for item in res_items.get('items', [])]
            
            if not video_ids:
                break

            req_videos = youtube.videos().list(part='snippet,statistics,contentDetails', id=','.join(video_ids))
            res_videos = req_videos.execute()
            
            for v in res_videos.get('items', []):
                v_stats = v['statistics']
                v_snippet = v['snippet']
                
                pub_date = datetime.fromisoformat(v_snippet['publishedAt'].replace('Z', '+00:00'))

                if pub_date < START_DATE:
                    stop_collecting = True
                    break

                if pub_date > END_DATE:
                    continue
                
                videos_data.append({
                    'channel_id': ch['channel_id'],
                    'channel_title': ch['channel_title'],
                    'video_id': v['id'],
                    'title': v_snippet['title'],
                    'description': v_snippet.get('description', '')[:500],
                    'published_at': v_snippet['publishedAt'],
                    'views': int(v_stats.get('viewCount', 0)),
                    'likes': int(v_stats.get('likeCount', 0)),
                    'comments': int(v_stats.get('commentCount', 0)),
                    'duration_iso': v['contentDetails'].get('duration', ''),
                    'tags': ", ".join(v_snippet.get('tags', []))
                })
                videos_count += 1
            
            next_page_token = res_items.get('nextPageToken')
            if not next_page_token:
                break
            
            time.sleep(0.2)
            
        except HttpError as e:
            print(f"Ошибка: {e}")
            break
            
    print(f"Собрано видео - {videos_count}.")

df_videos = pd.DataFrame(videos_data)
print(f"\nВсего видео собрано - {len(df_videos)}.")


Сбор видео за период c 25-10-2025 по 25-10-2026

Канал Кинопоиск. Собрано видео - 50.
Канал Глеб Соломин. Собрано видео - 28.
Канал Владимир Алипов. Собрано видео - 19.
Канал ПостНаука. Собрано видео - 96.
Канал Вселенная Плюс. Собрано видео - 22.
Канал Неземной подкаст Владимира Сурдина. Собрано видео - 34.
Канал Максим Кузнецов. Собрано видео - 712.
Канал Лютая физика. Собрано видео - 114.
Канал Научпок. Собрано видео - 17.
Канал Филолог всея Руси. Собрано видео - 83.
Канал Dameoz. Собрано видео - 38.
Канал Ирина Якутенко. Собрано видео - 101.
Канал Vert Dider. Собрано видео - 215.
Канал Redroom. Собрано видео - 16.
Канал Физика от Побединского. Собрано видео - 36.
Канал Правое полушарие Интроверта. Собрано видео - 264.
Канал Александр Мальков. Собрано видео - 332.
Канал ДОКТОРА. Собрано видео - 440.
Канал Ученые против мифов. Собрано видео - 26.
Канал Игорь Белецкий (investigator). Собрано видео - 34.
Канал ТОПЛЕС. Собрано видео - 34.
Канал Ганжа Глеб. Собрано видео - 85.
Канал Анд

In [8]:
#====================================ЭТАП 4: Обработка и сохранение данных=====================================================

def get_seconds(duration_iso):
    try:
        if pd.isna(duration_iso) or not isinstance(duration_iso, str) or duration_iso == '':
            return 0
        match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', duration_iso)
        if not match:
            return 0
        h, m, s = match.groups()
        return int(h or 0)*3600 + int(m or 0)*60 + int(s or 0)
    except Exception as e:
        print(f"⚠️ Ошибка при парсинге '{duration_iso}': {e}")
        return 0

# 1. Парсинг длительности и определение типа видео(обычное или шортс)
df_videos['duration_sec'] = df_videos['duration_iso'].apply(get_seconds)
df_videos['video_type'] = df_videos['duration_sec'].apply(lambda x: 'Shorts' if x <= 60 else 'Long Video')

# 2. Получение текущей даты по UTC
today_utc = datetime.now(timezone.utc).strftime('%Y-%m-%d')

# 3. Создание колонок и заполнение их строками 
df_channels['scraped_date'] = today_utc
df_videos['scraped_date'] = today_utc

# 4. Конвертация всех дат в datetime с UTC
df_videos['published_at'] = pd.to_datetime(df_videos['published_at'], utc=True)
df_channels['scraped_date'] = pd.to_datetime(df_channels['scraped_date'], utc=True)
df_videos['scraped_date'] = pd.to_datetime(df_videos['scraped_date'], utc=True)

# 5. Сохранение данных в Parquet (основной формат для анализа)
df_channels.to_csv(DATA_DIR / 'channels_data.csv', index=False, encoding='utf-8-sig')
df_videos.to_csv(DATA_DIR / 'videos_data.csv', index=False, encoding='utf-8-sig')

# Сохранение CSV с utf-8-sig для удобного открытия в Excel
df_channels.to_csv('channels_data.csv', index=False, encoding='utf-8-sig')
df_videos.to_csv('videos_data.csv', index=False, encoding='utf-8-sig')

print(f"\nВСЁ ГОТОВО!\nСохранены следующие данные:")
print(f"channels_data.parquet & videos_data.parquet")
print(f"channels_data.csv & videos_data.csv (для Excel)")
print(f"Данные актуальны на {END_DATE.strftime('%d-%m-%Y')}")


ВСЁ ГОТОВО!
Сохранены следующие данные:
channels_data.parquet & videos_data.parquet
channels_data.csv & videos_data.csv (для Excel)
Данные актуальны на 25-10-2026


In [9]:
print(f'\nДатафрейм df_channels, количество строк - {df_channels.shape[0]}, количество столбцов - {df_channels.shape[1]}')
print(f'Датафрейм df_videos, количество строк - {df_videos.shape[0]}, количество столбцов - {df_videos.shape[1]}', end='\n\n')

print('Названия столбцов df_channels:', *df_channels.columns, end='\n\n')
print('Названия столбцов df_videos:', *df_videos.columns, end='\n\n')


Датафрейм df_channels, количество строк - 40, количество столбцов - 7
Датафрейм df_videos, количество строк - 4157, количество столбцов - 14

Названия столбцов df_channels: channel_id channel_title subscribers total_views video_count uploads_playlist_id scraped_date

Названия столбцов df_videos: channel_id channel_title video_id title description published_at views likes comments duration_iso tags duration_sec video_type scraped_date

